In [3]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

ref_path = r"/home/gisuser/code/data/20-24_combined_rc.tif"
src_path = r"/home/gisuser/code/data/MB_c10_secFveg_2023_P.tif"
out_path = r"/home/gisuser/code/data/MB_c10_secFveg_2023_Pv2.tif"

In [4]:
with rasterio.open(ref_path) as ref:
    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_shape = (ref.height, ref.width)

with rasterio.open(src_path) as src:
    dst_data = np.empty(ref_shape, dtype=src.dtypes[0])
    reproject(
        source=src.read(1),
        destination=dst_data,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.nearest
    )

ref_profile.update(dtype=dst_data.dtype)
with rasterio.open(out_path, "w", **ref_profile) as dst:
    dst.write(dst_data, 1)

print("Done! Output:", out_path)

Done! Output: /home/gisuser/code/data/MB_c10_secFveg_2023_Pv2.tif


In [5]:
# Check to see if the files line up perfectly
with rasterio.open(ref_path) as ref, rasterio.open(out_path) as out:
    crs_match = ref.crs == out.crs
    transform_match = ref.transform == out.transform
    shape_match = (ref.height == out.height) and (ref.width == out.width)
    res_match = ref.res == out.res

    print("=== ALIGNMENT CHECK (ref vs output) ===")
    print(f"  CRS match: {crs_match}")
    print(f"  Transform match: {transform_match}")
    print(f"  Shape match: {shape_match}")
    print(f"  Pixel size match: {res_match}")

    if crs_match and transform_match and shape_match:
        print("PERFECTLY ALIGNED")
    else:
        print("NOT ALIGNED")

=== ALIGNMENT CHECK (ref vs output) ===
  CRS match: True
  Transform match: True
  Shape match: True
  Pixel size match: True
PERFECTLY ALIGNED
